In [1]:
import math
def pente(f, x, h=1e-5):
    return (f(x + h) - f(x - h)) / (2 * h)

print(f"pente(x**2, en 3) = {pente(lambda x: x ** 2, 3.0)}")

pente(x**2, en 3) = 6.000000000039306


In [2]:
class Value:
    def __init__(self, data, _prev=(), _op=""):
        self.grad = 0.0
        self._backward = lambda: None
        self.data = data
        self._prev = set(_prev)
        self._op = _op
        
    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), "+")

        def _backward():
            self.grad = out.grad
            other.grad = out.grad
        out._backward = _backward

        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), "*")

        def _backward():
            self.grad = other.data * out.grad
            other.grad = self.data * out.grad
        out._backward = _backward
        
        return out

    def __repr__(self):
        return f"Value(data={self.data})"
        

In [3]:
a = Value(2.0); b = Value(-3.0); c = Value(10.0)
# d = Value.__mul__(a, b)
d = a * b
# e = Value.__add__(d, c) 
e = d + c
f = Value(-2.0)
# L = Value.__mul__(e, f)
L = e * f

L.grad = 1.0
L._backward()
e._backward()
d._backward()

print(e)
print(e._op, sorted(v.data for v in e._prev))
print(d._op, sorted(v.data for v in d._prev))
print()
print(f"L {L.grad} | e {e.grad} | f {f.grad} | d {d.grad}")
print(f"c {c.grad} | a {a.grad} | b {b.grad}")

Value(data=4.0)
+ [-6.0, 10.0]
* [-3.0, 2.0]

L 1.0 | e -2.0 | f 4.0 | d -2.0
c -2.0 | a 6.0 | b -4.0


In [4]:
pente_capteur = pente(lambda av: ((av * -3.0) + 10.0) * -2.0, 2.0)
print(f"capteur sur a: {pente_capteur:.6f} | moteur : {a.grad}")

capteur sur a: 6.000000 | moteur : 6.0


In [5]:
# Verification du bug
a = Value(3.0)
s = a + a

s.grad = 1.0
s._backward()
print(f"moteur : a.grad = {a.grad}")
print(f"capteur : {pente(lambda av: av + av, 3.0):.4f}")

moteur : a.grad = 1.0
capteur : 2.0000


# Chapitre 4 — « Pousser » une valeur : comprendre le gradient et le backward

*Explication très détaillée, avec l'étal d'Ayélé. Pas de code du moteur : c'est à toi de l'écrire. Les seuls bouts de code ici sont des fonctions Python simples pour que tu voies « pousser » de tes yeux.*

---

## 0. Le mot qui bloque : « pousser »

Quand le livre (et moi) disons **« on pousse `a` d'un cheveu »**, ça veut dire une chose très simple :

> **On change UNE valeur d'entrée un tout petit peu, on ne touche à rien d'autre, on refait le calcul, et on regarde de combien le résultat a changé.**

Ce n'est pas un terme mathématique mystérieux. C'est une **expérience** : tu secoues une valeur et tu regardes ce que ça fait au résultat final.

### Les 3 gestes, toujours les mêmes

1. **Calcule normalement.** Note le résultat.
2. **Change une entrée** (et une seule) d'un tout petit peu.
3. **Recalcule et compare** : de combien le résultat a-t-il bougé ?

Le rapport « de combien le résultat a bougé / de combien j'ai changé l'entrée » s'appelle la **pente** (ou dérivée, ou gradient selon le contexte). C'est exactement ce que tu as fait avec Sètondji au chapitre 3.

### Pourquoi « un cheveu » et pas « 1 » ?

- Dans mes exemples ci-dessous, je pousse de **1** pour que les chiffres soient faciles à lire. Pour l'addition et la multiplication entre deux valeurs différentes, ça donne la pente exacte.
- Mais en toute rigueur, il faut une poussée **minuscule** (par exemple 0,001). Preuve avec `a × a` et `a = 3` :

| Poussée | Résultat | Pente mesurée |
|---|---|---|
| +1 (3 → 4) | 16 au lieu de 9 | 7 |
| +0,001 | 9,006001 | **6,001** |
| +0,000001 | 9,000006 | **6,000001** |

La vraie pente est **6**. Plus la poussée est petite, plus on s'en approche. C'est la différence entre « un petit pas » et « un cheveu ». Retiens : **la pente = ce que ça fait quand on pousse infiniment peu.**

---

## 1. L'exemple de référence : l'étal d'Ayélé

Ayélé vend des ananas. Voici son calcul :

- **dose** = 3 ananas
- **prix** = 250 FCFA par ananas
- **transport** = 100 FCFA
- **jours** = 2 (elle fait ça deux jours de suite)

```
coût  = dose × prix            = 3 × 250       = 750
total = coût + transport       = 750 + 100     = 850
L     = total × jours          = 850 × 2       = 1 700
```

**L** est le « résultat final » (l'équivalent de la loss dans un LLM). Notre but : savoir, pour **chaque entrée**, comment L réagit si on la pousse.

---

## 2. Pousser, une entrée à la fois (la méthode « à la main »)

Je pousse chaque entrée de **+1**, les autres ne bougent pas, et je refais tout le calcul.

### Je pousse `prix` : 250 → 251

```
coût  = 3 × 251     = 753    (avant : 750 → +3)
total = 753 + 100   = 853    (avant : 850 → +3)
L     = 853 × 2     = 1 706  (avant : 1 700 → +6)
```
**Résultat : L a bougé de +6.** Donc la pente de L par rapport à `prix` est **6**.
Lecture : « si l'ananas coûte 1 FCFA de plus, ma dépense sur les 2 jours monte de 6 FCFA ». Logique : 3 ananas × 2 jours = 6.

### Je pousse `dose` : 3 → 4

```
coût  = 4 × 250     = 1 000  (avant : 750 → +250)
total = 1 000 + 100 = 1 100  (avant : 850 → +250)
L     = 1 100 × 2   = 2 200  (avant : 1 700 → +500)
```
**L a bougé de +500.** Pente de L par rapport à `dose` : **500**.

### Je pousse `transport` : 100 → 101

```
coût  = 750          (inchangé : le transport n'y touche pas)
total = 750 + 101   = 851    (+1)
L     = 851 × 2     = 1 702  (+2)
```
**L a bougé de +2.** Pente : **2**.

### Je pousse `jours` : 2 → 3

```
coût  = 750, total = 850     (inchangés)
L     = 850 × 3     = 2 550  (avant : 1 700 → +850)
```
**L a bougé de +850.** Pente : **850**.

### Le tableau final (c'est le gradient de L)

| Entrée | Poussée | L bouge de | **Pente (grad)** |
|---|---|---|---|
| prix | +1 | +6 | **6** |
| dose | +1 | +500 | **500** |
| transport | +1 | +2 | **2** |
| jours | +1 | +850 | **850** |

**Le gradient, c'est ce tableau** : la liste des pentes, une par entrée. Rien de plus.

---

## 3. Le problème : cette méthode est trop lente

Pour obtenir ce tableau, j'ai refait **tout le calcul une fois par entrée**. Pour 4 entrées, ça va. Pour les 91 497 nombres de MiniLM, ça fait 91 497 recalculs complets (en vrai le double, on pousse des deux côtés) à chaque étape d'entraînement. Bien trop long.

Il faut obtenir **toutes les pentes en un seul parcours**, sans rien pousser. C'est ce que fait le **backward**.

---

## 4. L'idée clé : suivre la secousse à travers la chaîne

Reprenons `prix`. Quand je le pousse de 1, une « secousse » traverse le calcul :

```
prix  ──(+1)──►  coût  ──►  total  ──►  L
              ×dose=3      +transport    ×jours=2
              (+3)         (+3, inchangé) (+6)
```

- À travers `coût = dose × prix`, la secousse est **multipliée par `dose`** (3) : +1 devient +3.
- À travers `total = coût + transport`, la secousse **passe telle quelle** : +3 reste +3.
- À travers `L = total × jours`, la secousse est **multipliée par `jours`** (2) : +3 devient +6.

**Le total : 3 × 1 × 2 = 6.** La pente est le **produit des « amplifications »** le long du chemin.

Pareil pour `dose` : +1 devient +250 (× prix), reste +250 (addition), devient +500 (× jours) → 250 × 1 × 2 = **500** ✓.

### Les deux règles locales (les seules dont le moteur a besoin)

Chaque opération connaît **uniquement** sa propre amplification :

| Opération | Ce qu'elle fait à la secousse | Exemple |
|---|---|---|
| **Addition** `x + y` | La laisse passer **telle quelle** (× 1) aux deux entrées | `total = coût + transport` : pousser coût ou transport de 1 pousse total de 1 |
| **Multiplication** `x × y` | **Multiplie par la valeur de l'autre** | `coût = dose × prix` : pousser dose de 1 pousse coût de `prix` (250) ; pousser prix de 1 pousse coût de `dose` (3) |

Ces amplifications locales s'appellent des **dérivées locales**.

---

## 5. Le backward : faire la même chose, mais en partant de la fin

Au lieu de pousser chaque entrée et de regarder ce qui arrive à L, on **part de L et on remonte**, en répondant à : *« qui est responsable de ce qui arrive à L ? »*

On appelle **gradient reçu** d'un nœud : *de combien L bouge quand CE nœud bouge de 1.*

La règle de remontée est une seule ligne :

> **gradient du parent = gradient reçu × dérivée locale**

### Remontée sur l'étal d'Ayélé

| Étape | Qui parle | Calcul | Résultat |
|---|---|---|---|
| 0 | **L** | Point de départ : L bouge de 1 quand L bouge de 1 | `L.grad = 1` |
| 1 | L = total × jours (**multiplication**) | `total` reçoit 1 × `jours` ; `jours` reçoit 1 × `total` | `total.grad = 2`, `jours.grad = 850` |
| 2 | total = coût + transport (**addition**) | transmet tel quel : 2 × 1 | `coût.grad = 2`, `transport.grad = 2` |
| 3 | coût = dose × prix (**multiplication**) | `dose` reçoit 2 × `prix` ; `prix` reçoit 2 × `dose` | `dose.grad = 500`, `prix.grad = 6` |

On retrouve **exactement** le tableau de la section 2 : 6, 500, 2, 850. Mais cette fois **sans jamais rien pousser ni recalculer** : on a juste multiplié des petits nombres connus en remontant.

### Pourquoi à l'envers et pas à l'endroit ?

En remontant depuis L, **un seul parcours** donne la pente de **toutes** les entrées d'un coup. Dans un LLM, c'est ce qui rend l'entraînement possible : un backward donne les 91 497 pentes (ou les milliards d'un grand modèle) pour le prix d'un seul passage.

### Où est le « ticket » dont parle le livre ?

Au moment de l'**aller** (le calcul normal), chaque opération crée son résultat **et** prépare son mode d'emploi pour le retour : elle sait qui sont ses parents et quelles valeurs elle doit échanger (par exemple `dose` et `prix` pour la multiplication). Ce mode d'emploi est la fonction `_backward` accrochée à chaque nœud.

Deux choses à bien retenir :
- Un appel à `_backward` ne fait **qu'un seul pas** : il pousse le gradient de **sa** sortie vers **ses** parents. Il ne remonte pas tout seul.
- C'est pour ça qu'on les appelle **dans l'ordre inverse** (L, puis total, puis coût). Un nœud ne peut distribuer son gradient qu'après l'avoir **reçu**.

---

## 6. Le piège (section 5) : le nœud utilisé deux fois

### Le cas
Ayélé utilise **le même prix `p`** pour deux produits :

```
total = 3 × p  +  5 × p        avec p = 250   →   750 + 1 250 = 2 000
```

Le nœud `p` est utilisé **deux fois** dans le calcul.

### Pousser `p` de 1 (la vraie pente)

```
p = 251  →  3 × 251 + 5 × 251 = 753 + 1 255 = 2 008     (+8)
```

La pente est **8**. Ça se comprend très bien : la secousse arrive à `total` **par deux routes** :

```
          route 1 : ×3  ──► +3 ┐
p (+1) ──┤                       ├──► total : +3 +5 = +8
          route 2 : ×5  ──► +5 ┘
```

**Quand une secousse arrive par plusieurs routes, les effets s'additionnent.** (Comme un salaire qui augmente : ça agit sur le loyer, le transport, l'épargne, et l'effet total est la somme.)

### Le bug
Au retour, `p` reçoit **deux messages** de gradient : un de la route 1 (valeur 3), un de la route 2 (valeur 5). Si ton code **écrase** à chaque message (`grad = ...`), le second **remplace** le premier : `p.grad` vaut **5** au lieu de **8**. Faux, et **aucune erreur ne s'affiche**.

Même chose pour `s = a + a` dans le livre :
- vraie pente : 2 (pousse `a` de 1 : `s` monte de 2) ;
- moteur qui écrase : 1.

### La correction : accumuler
Chaque gradient qui arrive est **une contribution à ajouter** aux précédentes, pas la réponse finale. D'où :
- avant, on **écrasait** ;
- maintenant, on **additionne** à ce qu'il y avait déjà.

Et c'est pour ça que `grad` **démarre à 0.0** : c'est un **compteur de contributions**, et un compteur démarre à zéro. Si le premier message ajoute 3 à zéro puis le second ajoute 5, on obtient 8. ✓

### Pourquoi ce cas est partout
Dans un vrai réseau, un poids influence des dizaines de sorties à la fois : des chemins multiples par millions. Sans l'accumulation, aucun réseau ne s'entraînerait correctement.

### Ton test `L = a * a`
`a` est utilisé deux fois. Avec `a = 3` :
- chaque rôle de `a` envoie une contribution égale à la valeur de **l'autre** opérande, donc 3 ;
- total : 3 + 3 = **6** ✓ (la pente de a² est 2a = 6) ;
- un moteur qui écrase donnerait 3.

---

## 7. Le prix de l'accumulation : remettre à zéro

Puisque `grad` **accumule**, rien ne le vide entre deux calculs. Si tu fais deux `backward` à la suite sans remettre à zéro, les gradients du second **s'ajoutent aux restes du premier**. Les pentes deviennent **fausses, sans message d'erreur**.

**Réflexe à avoir :** remettre les gradients à zéro **avant chaque backward**. C'est précisément le rôle de la ligne `zero_grad` juste avant `loss.backward()` dans les 40 lignes du chapitre 1.

---

## 8. Essaie « pousser » toi-même (aucun autograd, juste Python)

Ce code **n'est pas le moteur du chapitre** : c'est le capteur du chapitre 3 appliqué à l'exemple d'Ayélé. Copie-le dans ton notebook, lance-le, et tu verras les pentes 6, 500, 2, 850 apparaître par simple poussée.

```python
def calcul(dose=3, prix=250, transport=100, jours=2):
    cout = dose * prix
    total = cout + transport
    return total * jours

base = calcul()
print("résultat de base :", base)

entrees = dict(dose=3, prix=250, transport=100, jours=2)
for nom, valeur in entrees.items():
    pousse = dict(entrees)
    pousse[nom] = valeur + 1          # on pousse UNE entrée de 1
    print(f"pousse {nom:9s} de 1 -> L bouge de {calcul(**pousse) - base}")
```

Sortie attendue :
```
résultat de base : 1700
pousse dose      de 1 -> L bouge de 500
pousse prix      de 1 -> L bouge de 6
pousse transport de 1 -> L bouge de 2
pousse jours     de 1 -> L bouge de 850
```

Et pour voir le piège du nœud utilisé deux fois :

```python
def deux_fois(p):
    return 3 * p + 5 * p

print(deux_fois(251) - deux_fois(250))   # donne 8, pas 5
```

Ces nombres sont **exactement** ceux que ton moteur d'autograd devra retrouver **sans** rien pousser. Quand ton `backward` donne les mêmes, tu sais qu'il est juste : c'est le *gradient checking* du chapitre.

---

## 9. Exercices (essaie avant de regarder les réponses)

Reprends l'étal d'Ayélé (dose 3, prix 250, transport 100, jours 2).

1. Si le transport passe de 100 à 130, de combien L bouge-t-il ? (Pense à `transport.grad`.)
2. Ayélé achète maintenant 4 ananas au lieu de 3. Sans refaire tout le calcul, de combien L bouge-t-il environ ? Est-ce exact ?
3. Pourquoi `jours.grad` vaut 850 et pas 2 ?
4. Un nouveau calcul : `L = x × x` avec `x = 5`. Que vaut `x.grad` avec un moteur correct ? Et avec un moteur qui écrase au lieu d'accumuler ?
5. Un nœud `q` est utilisé trois fois : `L = 2q + 3q + 4q`. Que vaut `q.grad` ?

### Réponses

1. +30 sur le transport donne +30 × 2 = **+60** (pente 2 × poussée 30). Exact ici, car la relation est « droite ».
2. +1 ananas → +500 environ. C'est exact : L passe de 1 700 à 2 200.
3. Parce que `jours` multiplie **tout le total** (850), pas seulement un petit morceau. Pousser `jours` de 1 ajoute un jour entier de dépense : 850. Les « 2 » sont la pente de `total`, pas de `jours`.
4. Moteur correct : **10** (5 + 5, la pente de x² est 2x). Moteur qui écrase : **5**.
5. **9** (2 + 3 + 4) : trois routes, les effets s'additionnent.

---

## 10. Mémo en 7 lignes

1. **Pousser** = changer UNE entrée un peu, refaire le calcul, regarder de combien le résultat bouge.
2. **Pente** = (variation du résultat) / (variation de l'entrée).
3. **Gradient** = la liste des pentes, une par entrée.
4. **Addition** : laisse passer la secousse telle quelle. **Multiplication** : multiplie par la valeur de l'autre.
5. **Backward** : on part de L (grad = 1) et on remonte avec « gradient du parent = gradient reçu × dérivée locale ».
6. **Plusieurs routes → on additionne** les contributions (d'où `grad` qui part de 0 et qui accumule).
7. **Accumuler → remettre à zéro** avant chaque nouveau backward.

# 5 -


In [11]:
a = Value(3.0)
s = a + a 

s.grad = 1.0
s._backward()
print(a.grad)

1.0
